# Exercise 1: The multipole zoo

**Block 1: Fields, kicks and linear optics**

**Learning objectives**
* Write the kick of any normal or skew multipole (dipole up to octupole) with one complex formula.
* Recognise each magnet from the pattern of its kicks in the transverse (x, y) plane.
* Learn that the kick of a 2(n+1)-pole grows like (amplitude)^n.

**Motivation.** An accelerator is built from a handful of magnet types. Each one acts on the
particles differently depending on the particles' amplitude. Being able to *see* the pattern
of kicks given by each magnet type, and to know how fast it grows with distance from the centre, 
is the first step to understanding which magnet causes which effect in phase space later.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

## Conventions

A particle travels along the beam axis $s$. Its transverse position is ($x$, $y$) and its
transverse normalised momenta (angles of the direction of motion w.r.t. the axis) are 
$\text{d}p_x = \text{d}x/\text{d}s$ and $\text{d}p_y = \text{d}y/\text{d}s$ (we use the small 
angle approximation, so the angle equals the transverse momentum divided by the total momentum).

A **thin magnet** does not move the particle; it only changes its angles by a *kick*
($\Delta p_x,\Delta p_y$). For all multipoles at once we use the same formula as MAD-X and Xsuite:

$$\Delta p_x - i\,\Delta p_y = -\sum_{n} \frac{k_nL + i\,k_{sn}L}{n!}\,(x + i y)^n$$

* *n* = 0 dipole, 1 quadrupole, 2 sextupole, 3 octupole (a 2(*n*+1)-pole).
* *knl[n]* = $k_nL$ is the **normal** integrated strength, *ksl[n]* the **skew** one.
* *i* is the imaginary unit. Complex numbers are just a compact way to handle *x* and *y* together.

The magnetic field is perpendicular to the kick: a kick in $+x$ needs a field in $−y$
(the Lorentz force $\mathbf{F}=q\mathbf{v}\times\mathbf{B}$). So $\Delta p_x\propto -B_y$ and 
$\Delta p_y\propto +B_x$.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()
NAMES = {0: "dipole", 1: "quadrupole", 2: "sextupole", 3: "octupole"}

## 1. The kick formula

**Task.** Implement `multipole_kick(x, y, n, k, skew)` using Python complex numbers
(`1j` is $i$; `z.real` and `z.imag` give the real and imaginary parts).
It must work on whole NumPy arrays at once.

In [ ]:
from math import factorial

def multipole_kick(x: np.ndarray, y: np.ndarray, n: int, k: float = 1.0,
                   skew: bool = False) -> tuple[np.ndarray, np.ndarray]:
    """Kick (dpx, dpy) of a single thin multipole of order n.

    Parameters
    ----------
    x, y : arrays of positions [m]
    n    : multipole order (0 dipole, 1 quad, 2 sextupole, 3 octupole)
    k    : integrated strength knl[n] (or ksl[n] if skew) [1/m^n]
    skew : False for a normal magnet, True for a skew one

    Convention:  dpx - i dpy = -(knl + i ksl) / n! * (x + i y)^n
    """
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")
    return dpx, dpy

In [ ]:
# --- self-check: compare with the toolkit ---
rng = np.random.default_rng(0)
pts = rng.uniform(-0.01, 0.01, (4, 100))
for n in range(4):
    for skew in (False, True):
        knl = [0] * n + ([0] if skew else [2.5])
        ksl = [0] * n + ([2.5] if skew else [0])
        ref = E.thin_multipole(pts, knl, ksl)
        dpx, dpy = multipole_kick(pts[0, :], pts[2, :], n, 2.5, skew)
        assert np.allclose(dpx, ref[1, :] - pts[1, :]), f"dpx wrong for n={n}, skew={skew}"
        assert np.allclose(dpy, ref[3, :] - pts[3, :]), f"dpy wrong for n={n}, skew={skew}"
print("GOOD! multipole_kick agrees with beamlib for all 8 magnets")

## 2. Do it by hand once

Expanding $(x+iy)^2 = x^2 - y^2 + 2ixy$ gives, for a **normal sextupole**
(strength $k_2L$):

$$\Delta p_x = -\frac{k_2L}{2}\,(x^2 - y^2), \qquad \Delta p_y = k_2L\,x\,y .$$

**Task.** Derive the same for a **normal octupole** and code both. 
Then check them against `multipole_kick`.

In [ ]:
def sextupole_by_hand(x, y, k2l):
    """Normal sextupole kick written out in x and y (no complex numbers)."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")


def octupole_by_hand(x, y, k3l):
    """Normal octupole kick written out in x and y (no complex numbers)."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
x, y = pts[:, 0], pts[:, 2]
assert np.allclose(sextupole_by_hand(x, y, 3.0), multipole_kick(x, y, 2, 3.0)), "sextupole formula wrong"
assert np.allclose(octupole_by_hand(x, y, 40.0), multipole_kick(x, y, 3, 40.0)), "octupole formula wrong"
print("GOOD! Hand-derived sextupole and octupole kicks are correct")

## 3. Kick maps

**Predict first.** Before running the next cell, sketch on paper the kick arrows of a
*normal quadrupole* with $k_1L$ > 0 at the four points (±1, 0), (0, ±1).
Which way do they point? Which plane is focused?

*Your prediction:*

In the plots below, the arrows show the **direction** of the kick (all drawn with the same
length) and the colour shows its **size**. Positions are in mm.

In [ ]:
def kick_map(ax, n, skew, title=True, grid=15, r=10e-3):
    g = np.linspace(-r, r, grid)
    X, Y = np.meshgrid(g, g)
    dpx, dpy = multipole_kick(X, Y, n, 1.0, skew)
    mag = np.hypot(dpx, dpy)
    with np.errstate(invalid="ignore", divide="ignore"):
        u, v = dpx / mag, dpy / mag
    ax.quiver(X * MM, Y * MM, u, v, np.log10(mag + 1e-30), cmap="viridis", pivot="mid",
              scale=grid * 1.3)
    ax.set_aspect("equal")
    ax.set_xlabel(r"$x$ [mm]")
    ax.set_ylabel(r"$y$ [mm]")
    if title:
        ax.set_title(f"{'skew' if skew else 'normal'} {NAMES[n]}")

fig, axs = plt.subplots(2, 4, figsize=(14, 7))
for n in range(4):
    kick_map(axs[0, n], n, skew=False)
    kick_map(axs[1, n], n, skew=True)
fig.suptitle("Kick direction (arrows) and size (colour, log scale; yellow = large)")
plt.show()

Things to notice:
* The **dipole** kicks every particle by the same amount: it steers the whole beam.
* The **quadrupole** kick is zero in the centre and grows linearly: it acts like a lens.
* A **skew** magnet is the normal one rotated by 90°/($n$+1): 90° for a dipole,
  45° for a quadrupole, 30° for a sextupole, 22.5° for an octupole.
* A normal 2($n$+1)-pole has 2($n$+1) directions where the kick points straight in or out.

Below: the field lines of a normal quadrupole (left) next to its kicks (right).
The field is everywhere perpendicular to the kick.

In [ ]:
g = np.linspace(-10e-3, 10e-3, 41)
X, Y = np.meshgrid(g, g)
dpx, dpy = multipole_kick(X, Y, 1, 1.0)
Bx, By = dpy, -dpx            # field direction (up to a positive factor)
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
axs[0].streamplot(X * MM, Y * MM, Bx, By, color=np.hypot(Bx, By), cmap="viridis", density=1.2)
axs[0].set_title("normal quadrupole: magnetic field lines")
kick_map(axs[1], 1, skew=False)
axs[1].set_title("normal quadrupole: kicks")
for ax in axs:
    ax.set_aspect("equal"); ax.set_xlabel(r"$x$ [mm]"); ax.set_ylabel(r"$y$ [mm]")
plt.show()

## 4. How fast does the kick grow?

The formula says $|\text{kick}|\propto|x+iy|^n=r^n$. On a log–log plot of |kick| against the distance
$r$ from the centre this is a straight line of slope $n$.

**Task.** Implement `measured_order(n, skew)`: compute the kick size at a few radii along any
direction you like, fit a straight line to log|kick| vs log $r$ (`np.polyfit`) and return the slope.

In [ ]:
def measured_order(n: int, skew: bool = False) -> float:
    """Slope of log|kick| versus log(r) for multipole order n."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
for n in range(4):
    for skew in (False, True):
        s = measured_order(n, skew)
        print(f"{'skew' if skew else 'normal':6s} {NAMES[n]:10s}: slope = {s:.6f}")
        assert abs(s - n) < 1e-6, "the slope should equal the multipole order n"
print("GOOD! The kick of a 2(n+1)-pole grows like r^n")

So at small amplitude the quadrupole dominates, and the higher multipoles matter more and
more as the particle moves away from the centre. Doubling the amplitude doubles a quadrupole
kick, but makes an octupole kick 8 times larger.

## 5. Challenge: which magnets are these?

The four kick maps below were made by four hidden magnets. For each one, give the order
(`"dipole"`, `"quadrupole"`, `"sextupole"`, `"octupole"`) and the type (`"normal"` or `"skew"`).

In [ ]:
import base64, json
_hidden = json.loads(base64.b64decode(
    "W1syLCAic2tldyJdLCBbMSwgIm5vcm1hbCJdLCBbMywgIm5vcm1hbCJdLCBbMSwgInNrZXciXV0="))  # no peeking!
fig, axs = plt.subplots(1, 4, figsize=(15, 4))
for i, (ax, (n, typ)) in enumerate(zip(axs, _hidden)):
    kick_map(ax, n, typ == "skew", title=False)
    ax.set_title(f"mystery magnet {i + 1}")
plt.show()

In [ ]:
# Fill in your answers, e.g. ("quadrupole", "normal")
my_answers = [(None, None), (None, None), (None, None), (None, None)]

In [ ]:
truth = [(NAMES[n], typ) for n, typ in _hidden]
n_right = sum(a == t for a, t in zip(my_answers, truth))
print(f"{n_right} / 4 correct")
assert n_right == 4, "not all right yet: look again at the number of 'radial' directions"
print("GOOD! Well done!")

## Reflection questions

1. Why does a skew quadrupole couple the horizontal and vertical motion, while a normal
   quadrupole does not? (Look at which kick component depends on which coordinate.)
2. A normal sextupole kicks a particle with $y\neq0$ but $x=0$. In which direction, and why
   does that matter for a beam that is not perfectly flat?
3. In a real machine all magnets have small unwanted higher-order components. Why do they
   mostly matter for particles at *large* amplitude?